# Document Classification: Naive Bayes vs KNN
## Using AG News Dataset

### Learning Objectives
- Understand how to convert text data into numerical vectors
- Implement document classification using Naive Bayes and KNN algorithms
- Compare performance and characteristics of both algorithms

### Dataset Information
**AG News Dataset**
- One of the most widely used benchmark datasets for text classification
- Contains news articles from over 2000 news sources
- 4 categories: World, Sports, Business, Science/Technology
- 120,000 training samples, 7,600 test samples

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

---
## 2. Theoretical Background

### 2.1 Naive Bayes Classifier

**Core Principle**
- A probabilistic classifier based on Bayes' theorem
- Assumes all features (words) are independent of each other ("Naive" assumption)

**Mathematical Formula**
$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$

Where:
- $P(C|X)$: Probability of category C given document X
- $P(X|C)$: Likelihood of document X in category C
- $P(C)$: Prior probability of category C

**Advantages**
- Very fast training speed
- Effective even with small datasets
- Performs well in high-dimensional spaces
- Provides probability scores for interpretability

**Disadvantages**
- Independence assumption is unrealistic
- Cannot capture relationships between words

### 2.2 K-Nearest Neighbors (KNN)

**Core Principle**
- A distance-based classification algorithm
- Classifies new data by majority vote of K nearest neighbors

**Distance Metric**
- Cosine similarity is commonly used for text data
$$\text{similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

**Advantages**
- Intuitive and easy to understand
- No training phase (Lazy Learning)
- Can model non-linear decision boundaries

**Disadvantages**
- Slow prediction time (computes distance to all training data)
- High memory usage
- Performance degrades in high dimensions (curse of dimensionality)
- Sensitive to the choice of K

---
## 3. Load AG News Dataset

In [ ]:
# Install datasets library if not already installed
# !pip install datasets

In [ ]:
from datasets import load_dataset

# Load AG News dataset
print("Loading AG News dataset...")
dataset = load_dataset('ag_news')

print(f"Dataset loaded successfully")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

In [ ]:
# Convert to pandas DataFrame for easier manipulation
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

# Map label numbers to category names
label_names = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
train_df['category'] = train_df['label'].map(label_names)
test_df['category'] = test_df['label'].map(label_names)

print("\nDataset structure:")
print(train_df.head())

In [ ]:
# For practical training time, we'll use a subset of the data
# Feel free to increase this for better performance
TRAIN_SIZE = 10000
TEST_SIZE = 2000

train_sample = train_df.sample(n=TRAIN_SIZE, random_state=42)
test_sample = test_df.sample(n=TEST_SIZE, random_state=42)

print(f"Using {TRAIN_SIZE} training samples and {TEST_SIZE} test samples")
print(f"\nCategory distribution in training set:")
print(train_sample['category'].value_counts())

---
## 4. Exploratory Data Analysis

In [ ]:
# Analyze text length
train_sample['word_count'] = train_sample['text'].str.split().str.len()

print("Text length statistics:")
print(train_sample['word_count'].describe())

In [ ]:
# Visualize category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category distribution
train_sample['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Category Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Text length distribution by category
train_sample.boxplot(column='word_count', by='category', ax=axes[1])
axes[1].set_title('Text Length by Category', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.show()

---
## 5. Prepare Train and Test Sets

In [ ]:
# Extract features and labels
X_train = train_sample['text']
y_train = train_sample['category']
X_test = test_sample['text']
y_test = test_sample['category']

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

---
## 6. Text Vectorization

### TF-IDF (Term Frequency-Inverse Document Frequency)
- **TF**: Frequency of a term in a document
- **IDF**: Rarity of a term across all documents
- **Result**: Higher weights for words that characterize specific documents

In [ ]:
# TF-IDF Vectorizer
vectorizer_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    stop_words='english'
)

print("Vectorizing text data...")
start = time()
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)
vectorize_time = time() - start

print(f"Vectorization completed in {vectorize_time:.2f} seconds")
print(f"\nVectorized shape: {X_train_tfidf.shape}")
print(f"Sparsity: {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])) * 100:.2f}%")

In [ ]:
# Count Vectorizer for comparison
vectorizer_count = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    stop_words='english'
)

X_train_count = vectorizer_count.fit_transform(X_train)
X_test_count = vectorizer_count.transform(X_test)

print(f"Count vectorization completed")
print(f"Shape: {X_train_count.shape}")

---
## 7. Naive Bayes Model

In [ ]:
# Naive Bayes with TF-IDF
print("Training Naive Bayes with TF-IDF...")
nb_tfidf = MultinomialNB(alpha=0.1)

start = time()
nb_tfidf.fit(X_train_tfidf, y_train)
train_time_nb_tfidf = time() - start

start = time()
y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)
predict_time_nb_tfidf = time() - start

acc_nb_tfidf = accuracy_score(y_test, y_pred_nb_tfidf)
f1_nb_tfidf = f1_score(y_test, y_pred_nb_tfidf, average='weighted')

print(f"Training time: {train_time_nb_tfidf:.4f}s")
print(f"Prediction time: {predict_time_nb_tfidf:.4f}s")
print(f"Accuracy: {acc_nb_tfidf:.4f} ({acc_nb_tfidf*100:.2f}%)")
print(f"F1-Score: {f1_nb_tfidf:.4f}")

In [ ]:
# Naive Bayes with Count Vectorizer
print("Training Naive Bayes with Count Vectorizer...")
nb_count = MultinomialNB(alpha=0.1)

start = time()
nb_count.fit(X_train_count, y_train)
train_time_nb_count = time() - start

start = time()
y_pred_nb_count = nb_count.predict(X_test_count)
predict_time_nb_count = time() - start

acc_nb_count = accuracy_score(y_test, y_pred_nb_count)
f1_nb_count = f1_score(y_test, y_pred_nb_count, average='weighted')

print(f"Training time: {train_time_nb_count:.4f}s")
print(f"Prediction time: {predict_time_nb_count:.4f}s")
print(f"Accuracy: {acc_nb_count:.4f} ({acc_nb_count*100:.2f}%)")
print(f"F1-Score: {f1_nb_count:.4f}")

---
## 8. KNN Model

In [ ]:
# KNN with different K values
k_values = [3, 5, 7]
knn_results = {}

for k in k_values:
    print(f"\nTraining KNN with k={k}...")
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine', n_jobs=-1)

    start = time()
    knn.fit(X_train_tfidf, y_train)
    train_time = time() - start

    start = time()
    y_pred = knn.predict(X_test_tfidf)
    predict_time = time() - start

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    knn_results[k] = {
        'accuracy': acc,
        'f1': f1,
        'train_time': train_time,
        'predict_time': predict_time,
        'predictions': y_pred
    }

    print(f"Training time: {train_time:.4f}s")
    print(f"Prediction time: {predict_time:.4f}s")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"F1-Score: {f1:.4f}")

---
## 9. Model Comparison

In [ ]:
# Create comparison dataframe
results_data = {
    'Model': ['NB + TF-IDF', 'NB + Count', 'KNN(k=3)', 'KNN(k=5)', 'KNN(k=7)'],
    'Accuracy': [
        acc_nb_tfidf,
        acc_nb_count,
        knn_results[3]['accuracy'],
        knn_results[5]['accuracy'],
        knn_results[7]['accuracy']
    ],
    'F1-Score': [
        f1_nb_tfidf,
        f1_nb_count,
        knn_results[3]['f1'],
        knn_results[5]['f1'],
        knn_results[7]['f1']
    ],
    'Train Time (s)': [
        train_time_nb_tfidf,
        train_time_nb_count,
        knn_results[3]['train_time'],
        knn_results[5]['train_time'],
        knn_results[7]['train_time']
    ],
    'Predict Time (s)': [
        predict_time_nb_tfidf,
        predict_time_nb_count,
        knn_results[3]['predict_time'],
        knn_results[5]['predict_time'],
        knn_results[7]['predict_time']
    ]
}

results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values('Accuracy', ascending=False)
print("\nPerformance Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy comparison
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
axes[0, 0].bar(results_data['Model'], results_data['Accuracy'], color=colors)
axes[0, 0].set_ylabel('Accuracy', fontweight='bold')
axes[0, 0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(results_data['Accuracy']):
    axes[0, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# F1-Score comparison
axes[0, 1].bar(results_data['Model'], results_data['F1-Score'], color=colors)
axes[0, 1].set_ylabel('F1-Score', fontweight='bold')
axes[0, 1].set_title('F1-Score Comparison', fontsize=14, fontweight='bold')
axes[0, 1].set_ylim([0, 1])
axes[0, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(results_data['F1-Score']):
    axes[0, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Training time comparison
axes[1, 0].bar(results_data['Model'], results_data['Train Time (s)'], color=colors)
axes[1, 0].set_ylabel('Time (seconds)', fontweight='bold')
axes[1, 0].set_title('Training Time Comparison', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)

# Prediction time comparison
axes[1, 1].bar(results_data['Model'], results_data['Predict Time (s)'], color=colors)
axes[1, 1].set_ylabel('Time (seconds)', fontweight='bold')
axes[1, 1].set_title('Prediction Time Comparison', fontsize=14, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 10. Detailed Performance Analysis

In [ ]:
# Classification report for best model
print("Classification Report (Naive Bayes + TF-IDF):")
print(classification_report(y_test, y_pred_nb_tfidf))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_nb_tfidf)
categories = sorted(train_sample['category'].unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix (Naive Bayes + TF-IDF)', fontweight='bold', fontsize=14)
plt.ylabel('True Label', fontweight='bold')
plt.xlabel('Predicted Label', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze misclassifications
misclassified_mask = y_test != y_pred_nb_tfidf
misclassified = test_sample[misclassified_mask].copy()
misclassified['predicted'] = y_pred_nb_tfidf[misclassified_mask]

print(f"\nMisclassified samples: {len(misclassified)} out of {len(y_test)} ({len(misclassified)/len(y_test)*100:.1f}%)")
print("\nExamples of misclassifications:")
for idx, row in misclassified.head(3).iterrows():
    print(f"\nText: {row['text'][:100]}...")
    print(f"True: {row['category']} | Predicted: {row['predicted']}")

---
## 11. Cross-Validation

In [ ]:
# 5-Fold Cross-Validation
print("Performing 5-fold cross-validation...\n")

# Prepare data for CV
X_all = vectorizer_tfidf.fit_transform(train_sample['text'])
y_all = train_sample['category']

# Naive Bayes CV
cv_scores_nb = cross_val_score(nb_tfidf, X_all, y_all, cv=5, scoring='accuracy')
print(f"Naive Bayes CV Scores: {cv_scores_nb}")
print(f"Mean: {cv_scores_nb.mean():.4f} (+/- {cv_scores_nb.std():.4f})")

# KNN CV (k=5)
knn_5 = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)
cv_scores_knn = cross_val_score(knn_5, X_all, y_all, cv=5, scoring='accuracy')
print(f"\nKNN (k=5) CV Scores: {cv_scores_knn}")
print(f"Mean: {cv_scores_knn.mean():.4f} (+/- {cv_scores_knn.std():.4f})")

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(10, 6))

folds = range(1, 6)
ax.plot(folds, cv_scores_nb, 'o-', linewidth=2, markersize=10, label='Naive Bayes')
ax.plot(folds, cv_scores_knn, 's-', linewidth=2, markersize=10, label='KNN (k=5)')

ax.axhline(y=cv_scores_nb.mean(), color='blue', linestyle='--', alpha=0.5, label=f'NB Mean: {cv_scores_nb.mean():.3f}')
ax.axhline(y=cv_scores_knn.mean(), color='orange', linestyle='--', alpha=0.5, label=f'KNN Mean: {cv_scores_knn.mean():.3f}')

ax.set_xlabel('Fold', fontweight='bold')
ax.set_ylabel('Accuracy', fontweight='bold')
ax.set_title('5-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 12. Predict New Documents

In [ ]:
# Example news articles for prediction
new_articles = [
    "Apple announces new iPhone with advanced AI capabilities and improved camera system.",
    "The stock market reached new highs today as investors showed confidence in economic recovery.",
    "Scientists discover a new planet in the habitable zone of a nearby star system.",
    "The national team won the championship after an exciting final match against their rivals."
]

# Vectorize new articles
new_articles_vec = vectorizer_tfidf.transform(new_articles)

# Predict with Naive Bayes
predictions = nb_tfidf.predict(new_articles_vec)
probabilities = nb_tfidf.predict_proba(new_articles_vec)

print("Prediction Results:\n")
for i, article in enumerate(new_articles):
    print(f"Article {i+1}: {article}")
    print(f"Predicted Category: {predictions[i]}")
    print("Probabilities:")
    for j, category in enumerate(nb_tfidf.classes_):
        print(f"  {category}: {probabilities[i][j]:.4f} ({probabilities[i][j]*100:.1f}%)")
    print()